# Data Quality Framework (v2 — Config-Driven)

A robust, config-driven Python SP that auto-scans all tables in a schema and runs data quality checks.

## Architecture
| Object | Purpose |
|---|---|
| `DQ_CONFIG` | Config table — defines which checks run, thresholds, regex, FK mappings, accepted values |
| `DQ_RUN_SUMMARY` | Run-level summary — pass rate, duration, tables scanned per execution |
| `DQ_CHECK_LOG` | Detailed results log — every check with PASS/FAIL, metric, details |

## Checks
| # | Check | Scope | Config-Driven |
|---|-------|-------|---|
| 1 | **Null Check** | All columns | Default (auto) |
| 2 | **Empty String Check** | Text columns | Default (auto) |
| 3 | **Duplicate Check** | All columns | Default (auto) |
| 4 | **Row Count Check** | Table-level | Default (auto) |
| 5 | **Row Count Anomaly** | Table-level | Compares vs previous run |
| 6 | **Freshness Check** | Date/Timestamp columns | Configurable threshold |
| 7 | **Future Date Check** | Date/Timestamp columns | Default (auto) |
| 8 | **Outlier Detection** | Numeric columns | Configurable std devs |
| 9 | **Regex Pattern Check** | Text columns | Config: email, phone, zip patterns |
| 10 | **Accepted Values** | Any column | Config: allowed value list |
| 11 | **Referential Integrity** | FK columns | Config: parent table + column mapping |
| 12 | **Cross-Column Rules** | Table-level | Config: SQL expression (e.g., `shipped_date >= order_date`) |
| 13 | **Column Profiling** | All columns | Stores min/max/avg/distinct per run |

In [ ]:
CREATE TABLE IF NOT EXISTS BRONZE_LAYER.SQLSERVER_SCH.DQ_CONFIG (
    CONFIG_ID         NUMBER AUTOINCREMENT,
    DATABASE_NAME     VARCHAR(256),
    SCHEMA_NAME       VARCHAR(256),
    TABLE_NAME        VARCHAR(256),
    COLUMN_NAME       VARCHAR(256),
    CHECK_TYPE        VARCHAR(100),
    IS_ACTIVE         BOOLEAN DEFAULT TRUE,
    THRESHOLD         FLOAT,
    REGEX_PATTERN     VARCHAR(1000),
    ACCEPTED_VALUES   VARCHAR(4000),
    FK_PARENT_TABLE   VARCHAR(512),
    FK_PARENT_COLUMN  VARCHAR(256),
    RULE_EXPRESSION   VARCHAR(4000),
    CREATED_AT        TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    UPDATED_AT        TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE BRONZE_LAYER.SQLSERVER_SCH.DQ_RUN_SUMMARY (
    RUN_BATCH_ID      VARCHAR(50),
    RUN_START         TIMESTAMP_NTZ,
    RUN_END           TIMESTAMP_NTZ,
    DURATION_SEC      FLOAT,
    DATABASE_NAME     VARCHAR(256),
    SCHEMA_NAME       VARCHAR(256),
    TABLES_SCANNED    NUMBER,
    TOTAL_CHECKS      NUMBER,
    PASSED            NUMBER,
    FAILED            NUMBER,
    PASS_RATE_PCT     FLOAT,
    STATUS            VARCHAR(20),
    TRIGGERED_BY      VARCHAR(256),
    TRIGGERED_ROLE    VARCHAR(256)
);

CREATE OR REPLACE TABLE BRONZE_LAYER.SQLSERVER_SCH.DQ_CHECK_LOG (
    RUN_BATCH_ID    VARCHAR(50),
    RUN_TIMESTAMP   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    DATABASE_NAME   VARCHAR(256),
    SCHEMA_NAME     VARCHAR(256),
    TABLE_NAME      VARCHAR(256),
    COLUMN_NAME     VARCHAR(256),
    CHECK_TYPE      VARCHAR(100),
    STATUS          VARCHAR(10),
    DETAILS         VARCHAR(4000),
    METRIC_VALUE    FLOAT,
    THRESHOLD       FLOAT,
    ROW_COUNT       NUMBER
);

In [ ]:
CREATE OR REPLACE PROCEDURE BRONZE_LAYER.SQLSERVER_SCH.SP_DATA_QUALITY_SCAN(
    P_DATABASE VARCHAR,
    P_SCHEMA   VARCHAR,
    P_FRESHNESS_HOURS FLOAT DEFAULT 24.0,
    P_OUTLIER_STDDEV  FLOAT DEFAULT 3.0
)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
EXECUTE AS CALLER
AS
$$
import snowflake.snowpark as snowpark
from datetime import datetime, date
import uuid

def run(session: snowpark.Session,
        P_DATABASE: str,
        P_SCHEMA: str,
        P_FRESHNESS_HOURS: float = 24.0,
        P_OUTLIER_STDDEV: float = 3.0) -> str:

    log_table = f"{P_DATABASE}.{P_SCHEMA}.DQ_CHECK_LOG"
    config_table = f"{P_DATABASE}.{P_SCHEMA}.DQ_CONFIG"
    summary_table = f"{P_DATABASE}.{P_SCHEMA}.DQ_RUN_SUMMARY"
    run_batch_id = str(uuid.uuid4())[:12]
    run_start = datetime.now()
    checks_run = 0
    passed = 0
    failed = 0
    tables_scanned = 0

    ctx = session.sql("SELECT CURRENT_USER() AS CU, CURRENT_ROLE() AS CR").collect()[0]
    triggered_by = ctx['CU']
    triggered_role = ctx['CR']

    cols_df = session.sql(f"""
        SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE
        FROM {P_DATABASE}.INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = '{P_SCHEMA}'
          AND TABLE_NAME NOT IN ('DQ_CHECK_LOG','DQ_CONFIG','DQ_RUN_SUMMARY','PIPELINE_CONTROL_TABLE')
        ORDER BY TABLE_NAME, ORDINAL_POSITION
    """).collect()

    table_cols = {}
    for r in cols_df:
        tbl = r['TABLE_NAME']
        if tbl not in table_cols:
            table_cols[tbl] = []
        table_cols[tbl].append((r['COLUMN_NAME'], r['DATA_TYPE']))

    config_rows = []
    try:
        config_rows = session.sql(f"""
            SELECT TABLE_NAME, COLUMN_NAME, CHECK_TYPE, THRESHOLD, REGEX_PATTERN,
                   ACCEPTED_VALUES, FK_PARENT_TABLE, FK_PARENT_COLUMN, RULE_EXPRESSION
            FROM {config_table}
            WHERE DATABASE_NAME = '{P_DATABASE}'
              AND SCHEMA_NAME = '{P_SCHEMA}'
              AND IS_ACTIVE = TRUE
        """).collect()
    except:
        pass

    config_lookup = {}
    for cr in config_rows:
        key = (cr['TABLE_NAME'], cr['COLUMN_NAME'] or '*', cr['CHECK_TYPE'])
        config_lookup[key] = cr

    def log_result(tbl, col_name, check_type, status, details, metric_val=None, threshold=None, row_count=None):
        nonlocal checks_run, passed, failed
        checks_run += 1
        if status == 'PASS':
            passed += 1
        elif status == 'FAIL':
            failed += 1
        metric_str = str(metric_val) if metric_val is not None else 'NULL'
        thresh_str = str(threshold) if threshold is not None else 'NULL'
        rc_str = str(row_count) if row_count is not None else 'NULL'
        details_escaped = details.replace("'", "''")
        session.sql(f"""
            INSERT INTO {log_table}
            (RUN_BATCH_ID, DATABASE_NAME, SCHEMA_NAME, TABLE_NAME, COLUMN_NAME, CHECK_TYPE, STATUS, DETAILS, METRIC_VALUE, THRESHOLD, ROW_COUNT)
            VALUES ('{run_batch_id}', '{P_DATABASE}', '{P_SCHEMA}', '{tbl}', '{col_name}', '{check_type}', '{status}',
                    '{details_escaped}', {metric_str}, {thresh_str}, {rc_str})
        """).collect()

    for tbl, columns in table_cols.items():
        fqn = f"{P_DATABASE}.{P_SCHEMA}.{tbl}"
        tables_scanned += 1

        rc = session.sql(f"SELECT COUNT(*) AS CNT FROM {fqn}").collect()[0]['CNT']
        if rc == 0:
            log_result(tbl, '*', 'ROW_COUNT', 'FAIL', 'Table has 0 rows', 0, 1, 0)
            continue
        else:
            log_result(tbl, '*', 'ROW_COUNT', 'PASS', f'Table has {rc} rows', rc, 1, rc)

        try:
            prev = session.sql(f"""
                SELECT ROW_COUNT FROM {log_table}
                WHERE TABLE_NAME = '{tbl}' AND CHECK_TYPE = 'ROW_COUNT' AND STATUS = 'PASS'
                  AND RUN_BATCH_ID != '{run_batch_id}'
                ORDER BY RUN_TIMESTAMP DESC LIMIT 1
            """).collect()
            if prev:
                prev_rc = prev[0]['ROW_COUNT']
                if prev_rc and prev_rc > 0:
                    pct_change = round(((rc - prev_rc) / prev_rc) * 100, 2)
                    threshold_pct = 50.0
                    if abs(pct_change) > threshold_pct:
                        log_result(tbl, '*', 'ROW_COUNT_ANOMALY', 'FAIL',
                                   f'Row count changed {pct_change}% (prev: {prev_rc}, curr: {rc})',
                                   abs(pct_change), threshold_pct, rc)
                    else:
                        log_result(tbl, '*', 'ROW_COUNT_ANOMALY', 'PASS',
                                   f'Row count change {pct_change}% within threshold',
                                   abs(pct_change), threshold_pct, rc)
        except:
            pass

        for col_name, data_type in columns:

            null_cnt = session.sql(f'SELECT COUNT(*) AS CNT FROM {fqn} WHERE "{col_name}" IS NULL').collect()[0]['CNT']
            null_pct = round((null_cnt / rc) * 100, 2) if rc > 0 else 0
            if null_cnt > 0:
                log_result(tbl, col_name, 'NULL_CHECK', 'FAIL',
                           f'{null_cnt} nulls ({null_pct}%)', null_pct, 0, null_cnt)
            else:
                log_result(tbl, col_name, 'NULL_CHECK', 'PASS', 'No nulls found', 0, 0, 0)

            if data_type in ('TEXT', 'VARCHAR', 'STRING', 'CHAR'):
                empty_cnt = session.sql(f"SELECT COUNT(*) AS CNT FROM {fqn} WHERE TRIM(\"{col_name}\") = '' OR \"{col_name}\" IS NULL").collect()[0]['CNT']
                net_empty = empty_cnt - null_cnt
                if net_empty > 0:
                    log_result(tbl, col_name, 'EMPTY_STRING', 'FAIL',
                               f'{net_empty} empty/blank strings found', net_empty, 0, net_empty)
                else:
                    log_result(tbl, col_name, 'EMPTY_STRING', 'PASS', 'No empty strings', 0, 0, 0)

            dup_cnt = session.sql(f"""
                SELECT COUNT(*) AS CNT FROM (
                    SELECT "{col_name}", COUNT(*) AS C FROM {fqn}
                    WHERE "{col_name}" IS NOT NULL
                    GROUP BY "{col_name}" HAVING COUNT(*) > 1
                )
            """).collect()[0]['CNT']
            if dup_cnt > 0:
                log_result(tbl, col_name, 'DUPLICATE_CHECK', 'FAIL',
                           f'{dup_cnt} duplicate values found', dup_cnt, 0, dup_cnt)
            else:
                log_result(tbl, col_name, 'DUPLICATE_CHECK', 'PASS', 'No duplicates', 0, 0, 0)

            if data_type in ('DATE', 'TIMESTAMP_NTZ', 'TIMESTAMP_LTZ', 'TIMESTAMP_TZ'):
                age_row = session.sql(f"""
                    SELECT ROUND(TIMESTAMPDIFF(SECOND, MAX("{col_name}")::TIMESTAMP_NTZ, CURRENT_TIMESTAMP()) / 3600, 2) AS AGE_H,
                           MAX("{col_name}") AS MX
                    FROM {fqn}
                """).collect()[0]
                age_hours = float(age_row['AGE_H']) if age_row['AGE_H'] is not None else None
                max_ts = age_row['MX']
                if age_hours is not None:
                    if age_hours > P_FRESHNESS_HOURS:
                        log_result(tbl, col_name, 'FRESHNESS', 'FAIL',
                                   f'Last value {max_ts}, {age_hours}h old (threshold: {P_FRESHNESS_HOURS}h)',
                                   age_hours, P_FRESHNESS_HOURS, None)
                    else:
                        log_result(tbl, col_name, 'FRESHNESS', 'PASS',
                                   f'Last value {max_ts}, {age_hours}h old', age_hours, P_FRESHNESS_HOURS, None)

                future_cnt = session.sql(f"SELECT COUNT(*) AS CNT FROM {fqn} WHERE \"{col_name}\"::TIMESTAMP_NTZ > CURRENT_TIMESTAMP()").collect()[0]['CNT']
                if future_cnt > 0:
                    log_result(tbl, col_name, 'FUTURE_DATE', 'FAIL',
                               f'{future_cnt} rows have future dates', future_cnt, 0, future_cnt)
                else:
                    log_result(tbl, col_name, 'FUTURE_DATE', 'PASS', 'No future dates', 0, 0, 0)

            if data_type in ('NUMBER', 'FLOAT', 'DECIMAL', 'INT', 'INTEGER', 'BIGINT', 'SMALLINT', 'DOUBLE'):
                stats = session.sql(f"""
                    SELECT AVG("{col_name}") AS MN, STDDEV("{col_name}") AS SD,
                           MIN("{col_name}") AS MINV, MAX("{col_name}") AS MAXV,
                           COUNT(DISTINCT "{col_name}") AS DIST
                    FROM {fqn} WHERE "{col_name}" IS NOT NULL
                """).collect()[0]
                mean_val, std_val = stats['MN'], stats['SD']
                log_result(tbl, col_name, 'COLUMN_PROFILE', 'INFO',
                           f'min={stats["MINV"]}, max={stats["MAXV"]}, avg={round(float(mean_val),2) if mean_val else None}, distinct={stats["DIST"]}',
                           stats['DIST'], None, rc)
                if mean_val is not None and std_val is not None and std_val > 0:
                    lower = float(mean_val) - P_OUTLIER_STDDEV * float(std_val)
                    upper = float(mean_val) + P_OUTLIER_STDDEV * float(std_val)
                    outlier_cnt = session.sql(f"""
                        SELECT COUNT(*) AS CNT FROM {fqn}
                        WHERE "{col_name}" IS NOT NULL
                          AND ("{col_name}" < {lower} OR "{col_name}" > {upper})
                    """).collect()[0]['CNT']
                    if outlier_cnt > 0:
                        log_result(tbl, col_name, 'OUTLIER', 'FAIL',
                                   f'{outlier_cnt} outliers outside {P_OUTLIER_STDDEV} std devs (range: {round(lower,2)} to {round(upper,2)})',
                                   outlier_cnt, P_OUTLIER_STDDEV, outlier_cnt)
                    else:
                        log_result(tbl, col_name, 'OUTLIER', 'PASS',
                                   f'No outliers outside {P_OUTLIER_STDDEV} std devs', 0, P_OUTLIER_STDDEV, 0)

            if data_type in ('TEXT', 'VARCHAR', 'STRING', 'CHAR'):
                prof = session.sql(f"""
                    SELECT MIN(LENGTH("{col_name}")) AS MINL, MAX(LENGTH("{col_name}")) AS MAXL,
                           ROUND(AVG(LENGTH("{col_name}")),1) AS AVGL, COUNT(DISTINCT "{col_name}") AS DIST
                    FROM {fqn} WHERE "{col_name}" IS NOT NULL
                """).collect()[0]
                log_result(tbl, col_name, 'COLUMN_PROFILE', 'INFO',
                           f'min_len={prof["MINL"]}, max_len={prof["MAXL"]}, avg_len={prof["AVGL"]}, distinct={prof["DIST"]}',
                           prof['DIST'], None, rc)

            cfg_regex = config_lookup.get((tbl, col_name, 'REGEX_PATTERN'))
            if cfg_regex and cfg_regex['REGEX_PATTERN']:
                pattern = cfg_regex['REGEX_PATTERN']
                pattern_escaped = pattern.replace("'", "''")
                regex_fail = session.sql(f"""
                    SELECT COUNT(*) AS CNT FROM {fqn}
                    WHERE "{col_name}" IS NOT NULL
                      AND NOT RLIKE("{col_name}", '{pattern_escaped}')
                """).collect()[0]['CNT']
                if regex_fail > 0:
                    log_result(tbl, col_name, 'REGEX_PATTERN', 'FAIL',
                               f'{regex_fail} values dont match pattern', regex_fail, 0, regex_fail)
                else:
                    log_result(tbl, col_name, 'REGEX_PATTERN', 'PASS',
                               f'All values match pattern', 0, 0, 0)

            cfg_av = config_lookup.get((tbl, col_name, 'ACCEPTED_VALUES'))
            if cfg_av and cfg_av['ACCEPTED_VALUES']:
                val_list = cfg_av['ACCEPTED_VALUES']
                av_fail = session.sql(f"""
                    SELECT COUNT(*) AS CNT FROM {fqn}
                    WHERE "{col_name}" IS NOT NULL
                      AND "{col_name}" NOT IN ({val_list})
                """).collect()[0]['CNT']
                if av_fail > 0:
                    log_result(tbl, col_name, 'ACCEPTED_VALUES', 'FAIL',
                               f'{av_fail} values outside accepted set: {val_list}', av_fail, 0, av_fail)
                else:
                    log_result(tbl, col_name, 'ACCEPTED_VALUES', 'PASS',
                               f'All values in accepted set', 0, 0, 0)

            cfg_fk = config_lookup.get((tbl, col_name, 'REFERENTIAL_INTEGRITY'))
            if cfg_fk and cfg_fk['FK_PARENT_TABLE'] and cfg_fk['FK_PARENT_COLUMN']:
                parent_tbl = cfg_fk['FK_PARENT_TABLE']
                parent_col = cfg_fk['FK_PARENT_COLUMN']
                orphan_cnt = session.sql(f"""
                    SELECT COUNT(*) AS CNT FROM {fqn} c
                    WHERE c."{col_name}" IS NOT NULL
                      AND c."{col_name}" NOT IN (SELECT "{parent_col}" FROM {parent_tbl})
                """).collect()[0]['CNT']
                if orphan_cnt > 0:
                    log_result(tbl, col_name, 'REFERENTIAL_INTEGRITY', 'FAIL',
                               f'{orphan_cnt} orphan records (not in {parent_tbl}.{parent_col})',
                               orphan_cnt, 0, orphan_cnt)
                else:
                    log_result(tbl, col_name, 'REFERENTIAL_INTEGRITY', 'PASS',
                               f'All values exist in {parent_tbl}.{parent_col}', 0, 0, 0)

        cross_col_configs = [(k, v) for k, v in config_lookup.items()
                             if k[0] == tbl and k[2] == 'CROSS_COLUMN_RULE']
        for cfg_key, cfg_val in cross_col_configs:
            rule_expr = cfg_val['RULE_EXPRESSION']
            if rule_expr:
                rule_fail = session.sql(f"""
                    SELECT COUNT(*) AS CNT FROM {fqn}
                    WHERE NOT ({rule_expr})
                """).collect()[0]['CNT']
                rule_escaped = rule_expr.replace("'", "''")
                if rule_fail > 0:
                    log_result(tbl, '*', 'CROSS_COLUMN_RULE', 'FAIL',
                               f'{rule_fail} rows violate rule: {rule_escaped}', rule_fail, 0, rule_fail)
                else:
                    log_result(tbl, '*', 'CROSS_COLUMN_RULE', 'PASS',
                               f'All rows satisfy rule: {rule_escaped}', 0, 0, 0)

    run_end = datetime.now()
    duration = round((run_end - run_start).total_seconds(), 2)
    pass_rate = round((passed / checks_run) * 100, 2) if checks_run > 0 else 0
    overall_status = 'PASSED' if failed == 0 else 'FAILED'

    session.sql(f"""
        INSERT INTO {summary_table}
        (RUN_BATCH_ID, RUN_START, RUN_END, DURATION_SEC, DATABASE_NAME, SCHEMA_NAME,
         TABLES_SCANNED, TOTAL_CHECKS, PASSED, FAILED, PASS_RATE_PCT, STATUS,
         TRIGGERED_BY, TRIGGERED_ROLE)
        VALUES ('{run_batch_id}', '{run_start.strftime("%Y-%m-%d %H:%M:%S")}',
                '{run_end.strftime("%Y-%m-%d %H:%M:%S")}', {duration},
                '{P_DATABASE}', '{P_SCHEMA}', {tables_scanned}, {checks_run},
                {passed}, {failed}, {pass_rate}, '{overall_status}',
                '{triggered_by}', '{triggered_role}')
    """).collect()

    return f'Run {run_batch_id}: {tables_scanned} tables, {checks_run} checks ({passed} passed, {failed} failed, {pass_rate}% pass rate) in {duration}s | triggered by {triggered_by} using role {triggered_role}'
$$;

In [ ]:
%%sql -r config_seed
TRUNCATE TABLE BRONZE_LAYER.SQLSERVER_SCH.DQ_CONFIG;

INSERT INTO BRONZE_LAYER.SQLSERVER_SCH.DQ_CONFIG
(DATABASE_NAME, SCHEMA_NAME, TABLE_NAME, COLUMN_NAME, CHECK_TYPE, REGEX_PATTERN)
VALUES
('BRONZE_LAYER','SQLSERVER_SCH','CUSTOMERS','EMAIL','REGEX_PATTERN','^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'),
('BRONZE_LAYER','SQLSERVER_SCH','STAFFS','EMAIL','REGEX_PATTERN','^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'),
('BRONZE_LAYER','SQLSERVER_SCH','STORES','EMAIL','REGEX_PATTERN','^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$'),
('BRONZE_LAYER','SQLSERVER_SCH','CUSTOMERS','PHONE','REGEX_PATTERN','^[(]?[0-9]{3}[)]?[-. ]?[0-9]{3}[-. ]?[0-9]{4}$');

INSERT INTO BRONZE_LAYER.SQLSERVER_SCH.DQ_CONFIG
(DATABASE_NAME, SCHEMA_NAME, TABLE_NAME, COLUMN_NAME, CHECK_TYPE, ACCEPTED_VALUES)
VALUES
('BRONZE_LAYER','SQLSERVER_SCH','ORDERS','ORDER_STATUS','ACCEPTED_VALUES','1,2,3,4'),
('BRONZE_LAYER','SQLSERVER_SCH','STAFFS','ACTIVE','ACCEPTED_VALUES','0,1');

INSERT INTO BRONZE_LAYER.SQLSERVER_SCH.DQ_CONFIG
(DATABASE_NAME, SCHEMA_NAME, TABLE_NAME, COLUMN_NAME, CHECK_TYPE, FK_PARENT_TABLE, FK_PARENT_COLUMN)
VALUES
('BRONZE_LAYER','SQLSERVER_SCH','ORDERS','CUSTOMER_ID','REFERENTIAL_INTEGRITY','BRONZE_LAYER.SQLSERVER_SCH.CUSTOMERS','CUSTOMER_ID'),
('BRONZE_LAYER','SQLSERVER_SCH','ORDERS','STORE_ID','REFERENTIAL_INTEGRITY','BRONZE_LAYER.SQLSERVER_SCH.STORES','STORE_ID'),
('BRONZE_LAYER','SQLSERVER_SCH','ORDERS','STAFF_ID','REFERENTIAL_INTEGRITY','BRONZE_LAYER.SQLSERVER_SCH.STAFFS','STAFF_ID'),
('BRONZE_LAYER','SQLSERVER_SCH','ORDER_ITEMS','ORDER_ID','REFERENTIAL_INTEGRITY','BRONZE_LAYER.SQLSERVER_SCH.ORDERS','ORDER_ID'),
('BRONZE_LAYER','SQLSERVER_SCH','ORDER_ITEMS','PRODUCT_ID','REFERENTIAL_INTEGRITY','BRONZE_LAYER.SQLSERVER_SCH.PRODUCTS','PRODUCT_ID'),
('BRONZE_LAYER','SQLSERVER_SCH','PRODUCTS','BRAND_ID','REFERENTIAL_INTEGRITY','BRONZE_LAYER.SQLSERVER_SCH.BRANDS','BRAND_ID'),
('BRONZE_LAYER','SQLSERVER_SCH','PRODUCTS','CATEGORY_ID','REFERENTIAL_INTEGRITY','BRONZE_LAYER.SQLSERVER_SCH.CATEGORIES','CATEGORY_ID'),
('BRONZE_LAYER','SQLSERVER_SCH','STOCKS','STORE_ID','REFERENTIAL_INTEGRITY','BRONZE_LAYER.SQLSERVER_SCH.STORES','STORE_ID'),
('BRONZE_LAYER','SQLSERVER_SCH','STOCKS','PRODUCT_ID','REFERENTIAL_INTEGRITY','BRONZE_LAYER.SQLSERVER_SCH.PRODUCTS','PRODUCT_ID');

INSERT INTO BRONZE_LAYER.SQLSERVER_SCH.DQ_CONFIG
(DATABASE_NAME, SCHEMA_NAME, TABLE_NAME, COLUMN_NAME, CHECK_TYPE, RULE_EXPRESSION)
VALUES
('BRONZE_LAYER','SQLSERVER_SCH','ORDERS','*','CROSS_COLUMN_RULE','"REQUIRED_DATE" >= "ORDER_DATE"'),
('BRONZE_LAYER','SQLSERVER_SCH','ORDERS','*','CROSS_COLUMN_RULE','"SHIPPED_DATE" IS NULL OR "SHIPPED_DATE" >= "ORDER_DATE"');

SELECT CHECK_TYPE, COUNT(*) AS RULES FROM BRONZE_LAYER.SQLSERVER_SCH.DQ_CONFIG GROUP BY CHECK_TYPE;

In [ ]:
%%sql -r dq_run_result
CALL BRONZE_LAYER.SQLSERVER_SCH.SP_DATA_QUALITY_SCAN(
    'BRONZE_LAYER',
    'SQLSERVER_SCH',
    24.0,
    3.0
);

In [ ]:
%%sql -r run_summary
SELECT * FROM BRONZE_LAYER.SQLSERVER_SCH.DQ_RUN_SUMMARY ORDER BY RUN_START DESC LIMIT 5;

In [ ]:
%%sql -r check_summary
SELECT CHECK_TYPE, STATUS, COUNT(*) AS CHECK_COUNT
FROM BRONZE_LAYER.SQLSERVER_SCH.DQ_CHECK_LOG
WHERE RUN_BATCH_ID = (SELECT RUN_BATCH_ID FROM BRONZE_LAYER.SQLSERVER_SCH.DQ_RUN_SUMMARY ORDER BY RUN_START DESC LIMIT 1)
GROUP BY CHECK_TYPE, STATUS
ORDER BY CHECK_TYPE, STATUS;

In [ ]:
%%sql -r failures_detail
SELECT TABLE_NAME, COLUMN_NAME, CHECK_TYPE, STATUS, DETAILS, METRIC_VALUE
FROM BRONZE_LAYER.SQLSERVER_SCH.DQ_CHECK_LOG
WHERE STATUS = 'FAIL'
  AND RUN_BATCH_ID = (SELECT RUN_BATCH_ID FROM BRONZE_LAYER.SQLSERVER_SCH.DQ_RUN_SUMMARY ORDER BY RUN_START DESC LIMIT 1)
ORDER BY CHECK_TYPE, TABLE_NAME, COLUMN_NAME;

In [ ]:
%%sql -r profiling_results
SELECT TABLE_NAME, COLUMN_NAME, CHECK_TYPE, DETAILS
FROM BRONZE_LAYER.SQLSERVER_SCH.DQ_CHECK_LOG
WHERE CHECK_TYPE = 'COLUMN_PROFILE'
  AND RUN_BATCH_ID = (SELECT RUN_BATCH_ID FROM BRONZE_LAYER.SQLSERVER_SCH.DQ_RUN_SUMMARY ORDER BY RUN_START DESC LIMIT 1)
ORDER BY TABLE_NAME, COLUMN_NAME;